# Лабораторная работа №2

### Классификация MNIST: MLP (scikit-learn) vs CNN (PyTorch LeNet)

**Цель работы:** Решить задачу классификации датасета MNIST используя MLP из scikit-learn и CNN (по типу LeNet) на PyTorch. Сравнить результаты по метрикам, сделать обоснованные выводы.

**Задачи:**
1. Загрузить и подготовить датасет MNIST
2. Обучить MLP классификатор из scikit-learn
3. Реализовать и обучить сверточную сеть LeNet на PyTorch
4. Сравнить метрики (accuracy) и время обучения
5. Сделать выводы о эффективности каждой модели

## 1. Импорт библиотек

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import time
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

## 2. Загрузка и подготовка данных

In [2]:
# Загрузка датасета MNIST
transform = transforms.ToTensor()
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Подготовка данных для MLP (преобразуем 28x28 в вектор 784)
X_train_mlp = train_data.data.numpy().reshape(-1, 28*28) / 255.0
y_train_mlp = train_data.targets.numpy()
X_test_mlp = test_data.data.numpy().reshape(-1, 28*28) / 255.0
y_test_mlp = test_data.targets.numpy()

# Подготовка DataLoader для CNN
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000, shuffle=False)

print(f"Обучающая выборка: {len(train_data)} изображений")
print(f"Тестовая выборка: {len(test_data)} изображений")

Обучающая выборка: 60000 изображений
Тестовая выборка: 10000 изображений


## 3. Модель 1: MLP из scikit-learn

In [3]:
# Создание и обучение MLP
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),  # два скрытых слоя
    max_iter=20,                    # количество эпох
    random_state=42,                # для воспроизводимости
    verbose=True                    # вывод прогресса
)

start_time = time.time()
mlp.fit(X_train_mlp, y_train_mlp)
mlp_time = time.time() - start_time

# Предсказание и оценка
y_pred_mlp = mlp.predict(X_test_mlp)
mlp_accuracy = accuracy_score(y_test_mlp, y_pred_mlp)

print(f"\nMLP - Время обучения: {mlp_time:.2f} сек")
print(f"MLP - Точность: {mlp_accuracy:.4f}")
print("\nОтчет классификации MLP:")
print(classification_report(y_test_mlp, y_pred_mlp))

Iteration 1, loss = 0.38502588
Iteration 2, loss = 0.14950580
Iteration 3, loss = 0.10273425
Iteration 4, loss = 0.07876612
Iteration 5, loss = 0.06287742
Iteration 6, loss = 0.05150452
Iteration 7, loss = 0.04168628
Iteration 8, loss = 0.03421156
Iteration 9, loss = 0.02852425
Iteration 10, loss = 0.02430944
Iteration 11, loss = 0.02026940
Iteration 12, loss = 0.01670841
Iteration 13, loss = 0.01354651
Iteration 14, loss = 0.01485205
Iteration 15, loss = 0.01153144
Iteration 16, loss = 0.01095233
Iteration 17, loss = 0.00810442
Iteration 18, loss = 0.00709180
Iteration 19, loss = 0.00439719
Iteration 20, loss = 0.00553457

MLP - Время обучения: 15.80 сек
MLP - Точность: 0.9769

Отчет классификации MLP:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       980
           1       0.99      0.99      0.99      1135
           2       0.98      0.96      0.97      1032
           3       0.96      0.99      0.97      1010
           4    

c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


## 4. Модель 2: CNN LeNet на PyTorch

In [4]:
# Определение архитектуры LeNet
class LeNet(nn.Module):
    # LeNet-5 адаптированный для MNIST (вход 28x28, выход 10 классов)
    def __init__(self):
        super(LeNet, self).__init__()
        # Сверточные слои
        self.conv1 = nn.Conv2d(1, 6, 5, padding=2)   # 1->6 каналов
        self.conv2 = nn.Conv2d(6, 16, 5)              # 6->16 каналов
        # Пулинг
        self.pool = nn.AvgPool2d(2, 2)
        # Полносвязные слои
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
        # Активация
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))   # 28x28 -> 14x14
        x = self.pool(self.relu(self.conv2(x)))   # 14x14 -> 5x5
        x = x.view(-1, 16 * 5 * 5)                # выпрямление
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Определение устройства
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nИспользуемое устройство: {device}")

# Инициализация модели
model = LeNet().to(device)
criterion = nn.CrossEntropyLoss()      # функция потерь
optimizer = optim.Adam(model.parameters(), lr=0.001)  # оптимизатор

# Обучение CNN
print("\nОбучение CNN...")
start_time = time.time()

for epoch in range(5):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Прямой проход
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Обратный проход
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Эпоха {epoch+1}/5, Потери: {running_loss/len(train_loader):.4f}")

cnn_time = time.time() - start_time

# Оценка CNN
model.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

cnn_accuracy = correct / total

print(f"\nCNN - Время обучения: {cnn_time:.2f} сек")
print(f"CNN - Точность: {cnn_accuracy:.4f}")
print("\nОтчет классификации CNN:")
print(classification_report(all_labels, all_preds))


Используемое устройство: cpu

Обучение CNN...
Эпоха 1/5, Потери: 0.3373
Эпоха 2/5, Потери: 0.0988
Эпоха 3/5, Потери: 0.0689
Эпоха 4/5, Потери: 0.0530
Эпоха 5/5, Потери: 0.0437

CNN - Время обучения: 29.88 сек
CNN - Точность: 0.9862

Отчет классификации CNN:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       980
           1       0.99      1.00      1.00      1135
           2       0.99      0.96      0.98      1032
           3       0.97      0.99      0.98      1010
           4       0.99      1.00      0.99       982
           5       0.99      0.98      0.99       892
           6       0.98      0.99      0.99       958
           7       0.98      0.98      0.98      1028
           8       0.97      0.99      0.98       974
           9       0.99      0.98      0.98      1009

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99    

## 5. Сравнение результатов

In [5]:
print("\n" + "="*60)
print("СРАВНЕНИЕ МОДЕЛЕЙ")
print("="*60)
print(f"{'Модель':<20} {'Точность':<15} {'Время обучения':<15}")
print("-"*50)
print(f"{'MLP (sklearn)':<20} {mlp_accuracy*100:.2f}%{'':<10} {mlp_time:.2f} сек")
print(f"{'CNN (LeNet)':<20} {cnn_accuracy*100:.2f}%{'':<10} {cnn_time:.2f} сек")
print("-"*50)
print(f"{'Разница CNN - MLP':<20} {(cnn_accuracy - mlp_accuracy)*100:+.2f}%{'':<10} {cnn_time - mlp_time:+.2f} сек")


СРАВНЕНИЕ МОДЕЛЕЙ
Модель               Точность        Время обучения 
--------------------------------------------------
MLP (sklearn)        97.69%           15.80 сек
CNN (LeNet)          98.62%           29.88 сек
--------------------------------------------------
Разница CNN - MLP    +0.93%           +14.07 сек


## 6. Выводы

**Анализ:**
1. CNN показывает более высокую точность, чем MLP, так как сверточные слои эффективно выделяют пространственные признаки изображений (края, углы, формы)
2. MLP обучается быстрее из-за более простой архитектуры, но уступает в качестве классификации
3. Для задачи распознавания рукописных цифр обе модели показывают хорошие результаты (>97%), но CNN дает преимущество в 0.5-1%

**Вывод:** Для классификации изображений CNN предпочтительнее MLP, так как обеспечивает более высокую точность за счет учета пространственной структуры данных. MLP можно использовать при ограниченных вычислительных ресурсах или для небольших датасетов.